# Лабораторна робота №2. Частина 2
## Наука про дані: Individual Household Electric Power Consumption

### Імпорт бібліотек

In [ ]:
import os
import zipfile
import urllib.request
import pandas as pd
import numpy as np
import timeit
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler

### Завантаження та читання датасету
Індивідуальне споживання електроенергії домогосподарства (UCI ML Repository).

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
ZIP_FILE = "household_power_consumption.zip"
CSV_FILE = "household_power_consumption.txt"

if not os.path.exists(CSV_FILE):
    if not os.path.exists(ZIP_FILE):
        print("Завантаження датасету...")
        urllib.request.urlretrieve(DATA_URL, ZIP_FILE)
        print("Завантажено.")
    
    print("Розпакування...")
    with zipfile.ZipFile(ZIP_FILE, "r") as z:
        z.extractall(".")
    print("Розпаковано.")
else:
    print(f"Файл {CSV_FILE} вже існує.")

df = pd.read_csv(
    CSV_FILE, sep=";", na_values=["?"],
    low_memory=False
)
print(f"Розмір: {df.shape}")
df.head()

### Data Cleaning
Очищення даних: обробка пропусків, приведення типів, створення datetime індексу.

In [ ]:
print("Пропуски до очищення:")
print(df.isnull().sum())
print(f"\nВідсоток пропусків: {df.isnull().sum().sum() / df.size * 100:.2f}%")

# Привести числові стовпці до numeric
numeric_cols = ["Global_active_power", "Global_reactive_power", "Voltage",
                "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Створити datetime стовпець
df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H:%M:%S")

# Заповнити пропуски медіаною
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Видалити рядки, де Date/Time не вдалося розпарсити
df = df.dropna(subset=["Datetime"])

# Додати допоміжні стовпці
df["Hour"] = df["Datetime"].dt.hour
df["DayOfWeek"] = df["Datetime"].dt.dayofweek
df["Month"] = df["Datetime"].dt.month

print(f"\nРозмір після очищення: {df.shape}")
print(f"Пропуски після очищення: {df.isnull().sum().sum()}")
df.dtypes

### Завдання 1: Записи з Global_active_power > 5 кВт
Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.

In [ ]:
def filter_high_power(df, threshold=5.0):
    """Обрати записи з Global_active_power > threshold кВт."""
    return df[df["Global_active_power"] > threshold].copy()

t = timeit.timeit(lambda: filter_high_power(df), number=10)
print(f"Час виконання (10 разів): {t:.4f} с")

high_power = filter_high_power(df)
print(f"\nЗаписів з потужністю > 5 кВт: {len(high_power)} ({len(high_power)/len(df)*100:.2f}%)")
print(f"Середня потужність у вибірці: {high_power['Global_active_power'].mean():.2f} кВт")
high_power.head(10)

### Завдання 2: Сила струму 19–20 А, пральна+холодильник > бойлер+кондиціонер
Обрати записи з Global_intensity в межах 19–20 А, серед них — ті, де Sub_metering_2 (пральна, сушарка, холодильник, освітлення) > Sub_metering_3 (бойлер, кондиціонер).

In [ ]:
def filter_current_and_metering(df):
    """Записи з силою струму 19-20 А, де Sub_metering_2 > Sub_metering_3."""
    current_mask = (df["Global_intensity"] >= 19) & (df["Global_intensity"] <= 20)
    current_df = df[current_mask]
    result = current_df[current_df["Sub_metering_2"] > current_df["Sub_metering_3"]]
    return result.copy()

t = timeit.timeit(lambda: filter_current_and_metering(df), number=10)
print(f"Час виконання (10 разів): {t:.4f} с")

current_filtered = filter_current_and_metering(df)
print(f"\nЗаписів з силою струму 19-20 А: {len(df[(df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20)])}")
print(f"З них Sub_metering_2 > Sub_metering_3: {len(current_filtered)}")
current_filtered.head(10)

### Завдання 3: Випадкова вибірка 500 000 записів
Обрати випадковим чином 500 000 записів (без повторів), обчислити середні величини трьох груп споживання.

In [ ]:
def random_sample_means(df, n=500000, seed=42):
    """Випадкова вибірка n записів, середні 3 груп споживання."""
    sample = df.sample(n=n, replace=False, random_state=seed)
    means = {
        "Sub_metering_1 (кухня)": sample["Sub_metering_1"].mean(),
        "Sub_metering_2 (пральня)": sample["Sub_metering_2"].mean(),
        "Sub_metering_3 (бойлер+конд.)": sample["Sub_metering_3"].mean(),
    }
    return sample, means

t = timeit.timeit(lambda: random_sample_means(df), number=10)
print(f"Час виконання (10 разів): {t:.4f} с")

sample, means = random_sample_means(df)
print(f"\nРозмір вибірки: {len(sample)}")
print("\nСередні величини споживання (Вт·год):")
for name, val in means.items():
    print(f"  {name}: {val:.4f}")

### Завдання 4: Записи після 18:00 з потужністю > 6 кВт
Обрати записи після 18:00, де середня потужність > 6 кВт за хвилину. Серед них — де група 2 (Sub_metering_2) є найбільшою. Потім: кожен 3-й з першої половини та кожен 4-й з другої половини.

In [ ]:
def filter_evening_high_power(df):
    """
    1) Записи після 18:00 з Global_active_power > 6 кВт
    2) Серед них — де Sub_metering_2 найбільша
    3) Кожен 3-й з першої половини, кожен 4-й з другої
    """
    # Крок 1: після 18:00, потужність > 6 кВт
    mask = (df["Hour"] >= 18) & (df["Global_active_power"] > 6)
    evening = df[mask].copy()
    
    # Крок 2: Sub_metering_2 найбільша серед трьох груп
    sm2_largest = evening[
        (evening["Sub_metering_2"] > evening["Sub_metering_1"]) &
        (evening["Sub_metering_2"] > evening["Sub_metering_3"])
    ].copy()
    
    # Крок 3: розбити навпіл
    mid = len(sm2_largest) // 2
    first_half = sm2_largest.iloc[:mid]
    second_half = sm2_largest.iloc[mid:]
    
    # Кожен 3-й з першої, кожен 4-й з другої
    result = pd.concat([first_half.iloc[::3], second_half.iloc[::4]])
    
    return evening, sm2_largest, result

t = timeit.timeit(lambda: filter_evening_high_power(df), number=10)
print(f"Час виконання (10 разів): {t:.4f} с")

evening, sm2_largest, final = filter_evening_high_power(df)
print(f"\nЗаписів після 18:00 з потужністю > 6 кВт: {len(evening)}")
print(f"З них Sub_metering_2 найбільша: {len(sm2_largest)}")
print(f"Фінальна вибірка (кожен 3-й + кожен 4-й): {len(final)}")
final.head(10)

### Нормалізація та стандартизація датасету
Min-Max нормалізація та Z-score стандартизація числових атрибутів.

In [ ]:
numeric_cols = ["Global_active_power", "Global_reactive_power", "Voltage",
                "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]

# Min-Max нормалізація
scaler_mm = MinMaxScaler()
df_normalized = df.copy()
df_normalized[numeric_cols] = scaler_mm.fit_transform(df[numeric_cols])

print("Min-Max нормалізація (перші 5 рядків):")
df_normalized[numeric_cols].head()

In [ ]:
# Z-score стандартизація
scaler_std = StandardScaler()
df_standardized = df.copy()
df_standardized[numeric_cols] = scaler_std.fit_transform(df[numeric_cols])

print("Z-score стандартизація (перші 5 рядків):")
df_standardized[numeric_cols].head()

In [ ]:
print("Порівняння статистик:")
print("\nОригінал:")
print(df[numeric_cols].describe().round(4).loc[["mean", "std", "min", "max"]])
print("\nНормалізовані (Min-Max):")
print(df_normalized[numeric_cols].describe().round(4).loc[["mean", "std", "min", "max"]])
print("\nСтандартизовані (Z-score):")
print(df_standardized[numeric_cols].describe().round(4).loc[["mean", "std", "min", "max"]])

### Коефіцієнти кореляції Пірсона та Спірмена
Для двох числових атрибутів: Global_active_power та Global_intensity.

In [ ]:
col1, col2 = "Global_active_power", "Global_intensity"

pearson_r, pearson_p = stats.pearsonr(df[col1], df[col2])
spearman_r, spearman_p = stats.spearmanr(df[col1], df[col2])

print(f"Кореляція між {col1} та {col2}:")
print(f"\n  Пірсон:  r = {pearson_r:.6f},  p-value = {pearson_p:.2e}")
print(f"  Спірмен: r = {spearman_r:.6f},  p-value = {spearman_p:.2e}")

t = timeit.timeit(lambda: stats.pearsonr(df[col1], df[col2]), number=10)
print(f"\nЧас Пірсона (10 разів): {t:.4f} с")

t = timeit.timeit(lambda: stats.spearmanr(df[col1], df[col2]), number=10)
print(f"Час Спірмена (10 разів): {t:.4f} с")

### One Hot Encoding категоріального атрибута
Створюємо категорію «час доби» (ніч, ранок, день, вечір) та виконуємо One Hot Encoding.

In [ ]:
def time_of_day(hour):
    if 6 <= hour < 12:
        return "ранок"
    elif 12 <= hour < 18:
        return "день"
    elif 18 <= hour < 23:
        return "вечір"
    else:
        return "ніч"

df["time_of_day"] = df["Hour"].apply(time_of_day)

print("Розподіл за часом доби:")
print(df["time_of_day"].value_counts())

# One Hot Encoding
df_encoded = pd.get_dummies(df, columns=["time_of_day"], prefix="tod")

print("\nНові стовпці після One Hot Encoding:")
tod_cols = [c for c in df_encoded.columns if c.startswith("tod_")]
print(tod_cols)
df_encoded[tod_cols].head(10)

### Профілювання часу основних операцій

In [ ]:
operations = {
    "Фільтр > 5 кВт": lambda: filter_high_power(df),
    "Фільтр 19-20 А + metering": lambda: filter_current_and_metering(df),
    "Випадкова вибірка 500k": lambda: random_sample_means(df),
    "Вечірній фільтр": lambda: filter_evening_high_power(df),
    "Пірсон": lambda: stats.pearsonr(df["Global_active_power"], df["Global_intensity"]),
    "Спірмен": lambda: stats.spearmanr(df["Global_active_power"], df["Global_intensity"]),
}

print("Профілювання часу виконання (середнє за 10 запусків):\n")
for name, func in operations.items():
    t = timeit.timeit(func, number=10)
    print(f"  {name:35s}: {t/10:.6f} с")